#### Tags

##### Data ingestion strategy:
<mark style="background: #D69AFE;">**MERGE**</mark>

##### Related pipeline:

**Load_Tags_E2E**

##### Source:

**Files** from FUAM_Lakehouse folder **bronze_file_location** variable

##### Target:

**1 Delta table** in FUAM_Lakehouse 
- **gold_table_name** variable value

In [ ]:
from pyspark.sql.functions import col, explode, to_date, date_format, lit
import pyspark.sql.functions as f
from delta.tables import *
import datetime

In [ ]:
## Parameters
display_data = True

In [ ]:
## Variables
bronze_file_location = f"Files/raw/tags/"
silver_table_name = "FUAM_Staging_Lakehouse.tags_silver"
gold_table_name = "tags"
gold_table_name_with_prefix = f"Tables/{gold_table_name}"

In [ ]:
# Clean Silver table, if exists
if spark.catalog.tableExists(silver_table_name):
    del_query = "DELETE FROM " + silver_table_name
    spark.sql(del_query)

In [ ]:
# Get Bronze data
bronze_df = spark.read.option("multiline", "true").json(bronze_file_location)

if display_data:
    display(bronze_df)

In [ ]:
# Explode json subset structure
exploded_df = bronze_df.select(explode("value").alias("d"))

# To prevent error in execution this stops the notebook in case no git connection is available
if exploded_df.count() == 0 :
    notebookutils.notebook.exit("No Tags available")

# Select all columns (columns are dynamic)
silver_df = exploded_df.select(
    col("d.*")
    )

# Convert key(s) to upper case
silver_df = silver_df.withColumn("tagId", f.upper(f.col("id")))
silver_df = silver_df.drop("id")

# Extract columns
silver_df = silver_df.withColumn("scope", f.col("scope.type"))
silver_df = silver_df.select('tagId', 'scope', 'displayName' )

# Select needed columns
#silver_df = silver_df.select("workspaceId", "repositoryName", "gitProviderType")

if display_data:
    display(silver_df)

In [ ]:
# Write prepared bronze_df to silver delta table
silver_df.write.mode("overwrite").option("mergeSchema", "true").format("delta").saveAsTable(silver_table_name)

if display_data:
    display(silver_df)

In [ ]:
# Merge data to gold table
silver_df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(gold_table_name)

In [ ]:
# write history of bronze files

mssparkutils.fs.cp(bronze_file_location, bronze_file_location.replace("Files/raw/", "Files/history/") + datetime.datetime.now().strftime('%Y/%m/%d') + "/", True)